In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go

from utils import *


In [ ]:
# these functions are also defined in the 10_neutral_drift_clustering.py script
def retrieve_dms_scores(dms_score_df):
    
    """
    Converts DMS dataframe into a dictionary where the key is a mutation
    and the value is the DMS enrichment score
    
    PARAMETERS
    ------------
    df: pandas dataframe
        dms results where column is original residue and index is mutation
    
    RETURNS
    ------------
    dms_dict: dictionary 
        dictionary where key is a mutation and the value is the DMS enrichment score
    
    """
    
    dms_dict = {
    f"{col}{idx}": dms_score_df.at[idx, col]
    for col in dms_score_df.columns
    for idx in dms_score_df.index
    }
    
    return(dms_dict)

def find_enriched_dms_bin(mut,
                          hi_bind_scores_dict,
                          wt_bind_scores_dict,
                          low_bind_scores_dict,
                          no_bind_scores_dict):

    """
    Given a mutation, returns the most enriched DMS bin. This is a variant 
    of the function defined in '10_neutral_drift_clustering.py' this version
    takes a single mutation whereas the other version takes a list of
    mutations
    
    PARAMETERS
    ------------
    mut str
        mutations to check
    hi_bind_scores_dict: dictionary
        maps mutations to enrichhment in high binding bin
    wt_bind_scores_dict: dictionary
        maps mutations to enrichhment in wild type binding bin
    lo_bind_scores_dict: dictionary
        maps mutations to enrichhment in low type binding bin
    no_bind_scores_dict: dictionary
        maps mutations to enrichhment in no type binding bin
        
    PARAMETERS
    ------------
    bin_loc: str 
        bin assignment
    """
    
    index_to_bin_dict = {0: 'No binding',
                         1: 'Low binding',
                         2: 'WT binding',
                         3: 'High binding'}
    hi_score_tmp = hi_bind_scores_dict.get(mut, np.nan)
    wt_score_tmp = wt_bind_scores_dict.get(mut, np.nan)
    lo_score_tmp = lo_bind_scores_dict.get(mut, np.nan)
    no_score_tmp = no_bind_scores_dict.get(mut, np.nan)

    scores_list_tmp = [no_score_tmp, lo_score_tmp, wt_score_tmp, hi_score_tmp]
    max_index = np.argmax(scores_list_tmp)

    bin_loc = index_to_bin_dict[max_index]

    if check_for_nan(scores_list_tmp):
        bin_loc = 'Not in DMS data'
        
    return(bin_loc)

def check_for_nan(list_to_check):
    
    """
    Checks whether a list contains an NaN value
    
    PARAMETERS
    ------------
    list_to_check: list
        list that will be checked for NaNs 
        
    PARAMETERS
    ------------
    True is list contains NaN
    False if list does NOT contain NaN
    """
    
    return(any(np.isnan(x) for x in list_to_check if isinstance(x, (int, float)))) 

def return_dms_scores(mut,
                          hi_bind_scores_dict,
                          wt_bind_scores_dict,
                          low_bind_scores_dict,
                          no_bind_scores_dict):

    """
    Given a mutation, returns the most enriched DMS bin
    
    PARAMETERS
    ------------
    mut str
        mutations to check
    hi_bind_scores_dict: dictionary
        maps mutations to enrichhment in high binding bin
    wt_bind_scores_dict: dictionary
        maps mutations to enrichhment in wild type binding bin
    lo_bind_scores_dict: dictionary
        maps mutations to enrichhment in low type binding bin
    no_bind_scores_dict: dictionary
        maps mutations to enrichhment in no type binding bin
        
    PARAMETERS
    ------------
    bin_loc: str 
        bin assignment
    """
    
#     dms_mutation_bin_list = []
    index_to_bin_dict = {0: 'No binding',
                         1: 'Low binding',
                         2: 'WT binding',
                         3: 'High binding'}
#     for mut in mut_list:
    hi_score_tmp = hi_bind_scores_dict.get(mut, np.nan)
    wt_score_tmp = wt_bind_scores_dict.get(mut, np.nan)
    lo_score_tmp = lo_bind_scores_dict.get(mut, np.nan)
    no_score_tmp = no_bind_scores_dict.get(mut, np.nan)

    scores_list_tmp = [no_score_tmp, lo_score_tmp, wt_score_tmp, hi_score_tmp]
    print(scores_list_tmp)
    return(scores_list_tmp)

In [ ]:
def mutate_seq(orig_aa_seq, 
               position, 
               new_aa):
    
    """
    mutates a specific position in an amino acid sequence.

    PARAMETERS
    -----------
    orig_aa_seq: str
        the original amino acid sequence (unmutated)
    position: int
        the position in the amino acid to be mutated (0-indexed)
    new_aa: str
        the amino acid to mutate to in the specified position 

    RETURNS
    -----------
    new_aa_seq: str
        the new, mutated amino acid sequence
        Note: if the position is out of range, orig_aa_seq will be returned 
    """
    
    position=position
    
    if 0 <= position < len(orig_aa_seq):
        new_aa_seq = list(orig_aa_seq)
        new_aa_seq[position] = new_aa
        new_aa_seq = "".join(new_aa_seq)
        return (new_aa_seq)
    else:
        print("Position out of range! Returning original sequence")
        return(orig_aa_seq)
    
def compute_class_change(mutation, row_data, df):
    
    """
    based on an input sequence (from row_data['aa_sequence']) this function 
    finds the same sequence without a given mutation by searching df. then, 
    a dictionary is created with various parameters of interest based on the 
    original (unmutated) and new (mutated) amino acid sequence
    
    PARAMETERS
    -----------
    mutation: str
        specific mutation being searched for, examples: A2V; R35K
    row_data: dictionary from iterrows
        row data obtained from looping through a dataframe with iterrows
    df: pandas dataframe
        original dataframe with all sequence and scores
        
    RETURNS
    -----------
    original_versus_new_dict: dictionary
        dictionary with information from original and new amino acid sequence. 
        this should be appended to a dict-list and then eventually used to make
        a pandas dataframe.
        NOTE: this function returns None if the mutation is not found in the 
        "new" amino acid sequence. 
    """ 
    
    original_aa, new_aa = mutation[0], mutation[-1]
    position = int(mutation[1:-1]) - 1  # Convert to zero-indexed

    aa_seq_new = row_data['aa_sequence']
    aa_mutations_new = row_data['aa_mutations']
    score_mean_new = row_data[['normalized_score_rep1', 'normalized_score_rep2']].mean()
    class_new = row_data['predicted_category']
    class_val_new = class_values_dict[class_new]

    if aa_seq_new[position] == new_aa:
        aa_seq_orig = mutate_seq(aa_seq_new, position, original_aa)
        aa_seq_orig_info_df = df.loc[df['aa_sequence'] == aa_seq_orig]

        if not aa_seq_orig_info_df.empty:
            class_orig = aa_seq_orig_info_df.at[aa_seq_orig_info_df.index[0], 'predicted_category']
            class_val_orig = class_values_dict[class_orig]
            aa_mutations_orig = aa_seq_orig_info_df.at[aa_seq_orig_info_df.index[0], 'aa_mutations']
            
            score_rep1_orig, score_rep2_orig = aa_seq_orig_info_df.iloc[0][['normalized_score_rep1', 'normalized_score_rep2']]
            score_mean_orig = np.mean([score_rep1_orig, score_rep2_orig])
            delta_score = score_mean_new - score_mean_orig
            original_versus_new_dict = {
                'original_seq': aa_seq_orig,
                'original_mutations': aa_mutations_orig,
                'new_seq': aa_seq_new,
                'new_mutations': aa_mutations_new,
                'mutation': mutation,
                'class_orig': class_orig,
                'class_new': class_new,
                'class_change': f"{class_orig}_to_{class_new}",
                'class_change_value': class_val_new - class_val_orig,
                'new_score': score_mean_new,
                'orig_score': score_mean_orig,
                'delta_score': delta_score
            }
            return(original_versus_new_dict)
    return None

In [ ]:
def interactive_slopegraph(df, current_mutation, outpath, dms_bin, orig_col='orig_score', new_col='new_score', 
                           class_orig='class_orig', class_new='class_new', 
                           change_col='class_change_value', hover_cols=None, save=False):
    """
    Creates an interactive plot comparing normalized binding scores with and without
    a mutation of interest (i.e., current_mutation)
    
    PARAMETERS
    -----------
    df: pandas dataframe
        dataframe with data about sequence with and without mutations. important
        columns include: orig_score, new_score, class_orig, class_new, and class_change_value
    orig_col : str
        column name containing original scores
    new_col : str
        column name containing new scores for sequence with a mutation
    class_orig : str
        column name indicating original class (WT-like, Active, Inactive)
    class_new : str
        column name indicating new class (WT-like, Active, Inactive)
    change_col : str
        column quantifying class change (based on WT-like:2, Active:1, Inactive:0)
    hover_cols : list
        additional columns to display in the hover tooltip.
    save : boolean
        indicate whether to save figures
    outpath : str
        path for output figure
        
    RETURNS
    -----------
        NONE - just plots
    """
    hover_cols = hover_cols or []  # default to empty list if not provided
    
    # define color mapping for class labels
    class_colors = {'WT-like': 'orange', 
                    'Active': '#3B71B2', 
                    'Inactive': 'black'}

    # define line width mapping for absolute change values
    linewidth_map = {0: 0.5, 
                     1: 0.5, #2
                     2: 0.5} #5

    fig = go.Figure()
    fig.update_layout(
    width=600,  # Adjust this value to your preference
    height=500,  # Optional: adjust height if needed
)

    for _, row in df.iterrows():
        abs_change = abs(row[change_col])  # Get absolute value of change
        line_width = linewidth_map.get(abs_change, 0.5)  # Default to smallest width

        # Construct hover text for extra columns
        hover_text = "<br>".join([f"{col}: {row[col]}" for col in hover_cols])

        # Line connecting scores
        fig.add_trace(go.Scatter(
            x=[0, 1], y=[row[orig_col], row[new_col]],
            mode='lines',
            line=dict(color='gray', width=line_width),
            hoverinfo='skip'  # Hide hover for lines
        ))

        # Scatter for original score
        fig.add_trace(go.Scatter(
            x=[0], y=[row[orig_col]],
            mode='markers',
            marker=dict(color=class_colors.get(row[class_orig], 'gray'), size=12, line=dict(color='black', width=0.5)),
            name=row[class_orig],
            hovertemplate=f"{orig_col}: {row[orig_col]}<br>Class: {row[class_orig]}<br>{hover_text}<extra></extra>"
        ))

        # Scatter for new score
        fig.add_trace(go.Scatter(
            x=[1], y=[row[new_col]],
            mode='markers',
            marker=dict(color=class_colors.get(row[class_new], 'gray'), size=12, line=dict(color='black', width=0.5)),
            name=row[class_new],
            hovertemplate=f"{new_col}: {row[new_col]}<br>Class: {row[class_new]}<br>{hover_text}<extra></extra>"
        ))

    # Layout adjustments to remove gridlines
    fig.update_layout(
    title=f"{current_mutation} ({dms_bin})",
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=[f"Without {current_mutation}", f"With {current_mutation}"],
        showgrid=False,  
        zeroline=False
    ),
    yaxis=dict(
        title="Normalized binding score", 
        zeroline=True,   # Ensures the zero line is present
        range=[-0.05, 0.85],  
        tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1.0],  # Add custom y-axis tick marks
        tickmode='array',  
        showgrid=True,    # Optional: Adds faint gridlines for reference
    ),
    showlegend=False,
    template="plotly_white"
)


    # Show the figure
    fig.show()

    if save:
        fig.write_image(outpath+'.png', scale=3)
        fig.write_image(outpath+'.pdf', scale=3)


In [ ]:
norm_counts_annotated_path = '../../minibinders_orthorep_data/minibinders_orthorep_outputs/maa_003/neutral_drift_library/library_scores.csv'
norm_counts_annotated_df = pd.read_csv(norm_counts_annotated_path, index_col=0)
outdir = '../../minibinders_orthorep_data/minibinders_orthorep_outputs/maa_003/neutral_drift_epistasis/'
make_dir(outdir)


In [ ]:
all_mutations = [(extract_mutations_from_string(mutations)) \
               for mutations in norm_counts_annotated_df['aa_mutations']]

all_mutations = set(flatten_mutations(all_mutations))
all_mutations.remove('')

In [ ]:
# assigning values to each class, to make it easy to identify class switches 
class_values_dict = {'WT-like': 2, 
                     'Active': 1, 
                     'Inactive': 0}

class_change_dict_list = [] # list that will be used to convert dictionaries to a dataframe

for mutation in all_mutations:
    sequences_with_mutation_df = norm_counts_annotated_df.loc[
        norm_counts_annotated_df['aa_mutations'].str.contains(mutation)
    ].reset_index(drop=True) # reset_index should make a copy, so original dataframe is not altered

    for _, row in sequences_with_mutation_df.iterrows():
        class_change_info = compute_class_change(mutation, row, norm_counts_annotated_df)
        if class_change_info:
            class_change_dict_list.append(class_change_info)
        else: 
            pass

In [ ]:
hi_bind_dms_path = '../data/dms/mb376/mb376_dms_hi-bind.xlsx'
wt_bind_dms_path = '../data/dms/mb376/mb376_dms_wt-bind.xlsx'
lo_bind_dms_path = '../data/dms/mb376/mb376_dms_lo-bind.xlsx'
no_bind_dms_path = '../data/dms/mb376/mb376_dms_no-bind.xlsx'

hi_bind_dms_df = pd.read_excel(hi_bind_dms_path, index_col=0)
wt_bind_dms_df = pd.read_excel(wt_bind_dms_path, index_col=0)
lo_bind_dms_df = pd.read_excel(lo_bind_dms_path, index_col=0)
no_bind_dms_df = pd.read_excel(no_bind_dms_path, index_col=0)

original_residues = [residue[0]+str(aa_no) for aa_no, residue in enumerate(hi_bind_dms_df.columns.to_list(),1)]
hi_bind_dms_df = hi_bind_dms_df.rename(columns=dict(zip(hi_bind_dms_df.columns.to_list(), original_residues)))
wt_bind_dms_df = wt_bind_dms_df.rename(columns=dict(zip(wt_bind_dms_df.columns.to_list(), original_residues)))
lo_bind_dms_df = lo_bind_dms_df.rename(columns=dict(zip(lo_bind_dms_df.columns.to_list(), original_residues)))
no_bind_dms_df = no_bind_dms_df.rename(columns=dict(zip(no_bind_dms_df.columns.to_list(), original_residues)))

hi_bind_scores_dict = retrieve_dms_scores(hi_bind_dms_df)
wt_bind_scores_dict = retrieve_dms_scores(wt_bind_dms_df)
lo_bind_scores_dict = retrieve_dms_scores(lo_bind_dms_df)
no_bind_scores_dict = retrieve_dms_scores(no_bind_dms_df)

In [ ]:
class_change_info_df = pd.DataFrame(class_change_dict_list)
muts_with_class_switches_df = pd.DataFrame()
muts_without_class_switches_df = pd.DataFrame()

unique_muts_to_check = set(class_change_info_df['mutation'].to_list())
muts_with_class_switches_list = []
muts_without_class_switches_list = []
for mut in unique_muts_to_check:
    seqs_with_mut_tmp = class_change_info_df.copy()[class_change_info_df['mutation']==mut]
    unique_switches = len(set(seqs_with_mut_tmp['class_change_value']))
    if unique_switches > 1:
        muts_with_class_switches_list.append(mut)
        muts_of_interest_df = class_change_info_df.loc[class_change_info_df['mutation'] == mut].copy()
        muts_with_class_switches_df = pd.concat([muts_with_class_switches_df,muts_of_interest_df])
        muts_with_class_switches_df.reset_index(drop=True, inplace=True)
    else:
        muts_without_class_switches_list.append(mut)
        muts_of_interest_2_df = class_change_info_df.loc[class_change_info_df['mutation'] == mut].copy()
        muts_without_class_switches_df = pd.concat([muts_without_class_switches_df,muts_of_interest_2_df])
        muts_without_class_switches_df.reset_index(drop=True, inplace=True)

In [ ]:
for current_mut in set(muts_with_class_switches_df['mutation']):
    dms_bin = find_enriched_dms_bin(current_mut,
                                   hi_bind_scores_dict,
                                   wt_bind_scores_dict,
                                   lo_bind_scores_dict,
                                   no_bind_scores_dict)
    outpath = f"{outdir}{current_mut}"
    df_test = muts_with_class_switches_df.loc[muts_with_class_switches_df['mutation']==current_mut].copy()
    interactive_slopegraph(df_test, current_mutation=current_mut, outpath=outpath, dms_bin= dms_bin,
                           hover_cols=['class_change', 'mutation', 'delta_score'],save=True)

muts_with_class_switches_df.to_csv('../../minibinders_orthorep_data/minibinders_orthorep_outputs/maa_003/neutral_drift_epistasis/class_switch_data.csv')